# Preprocessing Data Sensor Upstream untuk Temporal Fusion Transformer (TFT)

Notebook ini adalah versi eksploratif dari `preprocessing.py`. Fokusnya adalah:
1. Menjalankan tahap-tahap preprocessing yang sama seperti `preprocessing.py` (dengan mengimpor fungsinya langsung), sambil menampilkan hasil antara di tiap tahap.
2. Menambahkan visualisasi EDA (missing value, distribusi, plot time series, korelasi antar parameter) yang tidak ada di versi script.

**Scope notebook ini hanya sampai data siap pakai** (parquet + metadata.json). Pembangunan `TimeSeriesDataSet` dan training TFT dilakukan di script/notebook model yang terpisah.

> Pastikan `preprocessing.py` berada di folder yang sama dengan notebook ini, karena beberapa cell akan `import` fungsi darinya.

# Import & Konfigurasi

In [ ]:
import sys
sys.path.append(".")

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from preprocessing import (
    Config,
    load_data,
    parse_and_sort,
    drop_duplicates,
    resample_per_group,
    filter_sparse_groups,
    handle_outliers,
    impute_missing,
    add_time_idx,
    add_calendar_features,
    compute_split_cutoffs,
    make_reference_split,
    save_outputs,
    KNOWN_CALENDAR_FEATURES,
)

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (12, 4)

In [ ]:
# Sesuaikan konfigurasi berikut dengan kondisi data CSV kamu.

cfg = Config(
    input_path="data/sensor_pep_prod.csv",
    output_dir="output/",
    datetime_col="timestamp",
    group_cols=["well_id"],                   # bisa lebih dari satu, mis. ["field", "well_id"]
    target_cols=["pressure", "flowrate", "temperature", "vibration"],
    static_categoricals=[],                    # mis. ["field_name", "equipment_type"] kalau ada
    freq="h",                                  # frekuensi resample: "h"=jam, "D"=hari, "15min", dst.
    max_missing_ratio=0.4,
    outlier_method="iqr",
    outlier_threshold=3.0,
    train_ratio=0.7,
    val_ratio=0.15,
    max_encoder_length=168,     # panjang histori (lookback) untuk TFT
    max_prediction_length=24,   # panjang horizon forecast
)
cfg

# Load Data

In [ ]:
# --- OPSIONAL: bikin data sintetis untuk uji coba alur notebook ---
# Skip cell ini kalau kamu sudah punya data asli di cfg.input_path

import os
os.makedirs("data", exist_ok=True)

if not os.path.exists(cfg.input_path):
    np.random.seed(42)
    rows = []
    for well in ["WELL_A", "WELL_B", "WELL_C"]:
        ts = pd.date_range("2024-01-01", periods=24 * 90, freq="h")  # 90 hari data jam-jaman
        ts = ts.delete(np.random.choice(len(ts), 40, replace=False))  # simulasi gap
        for t in ts:
            pressure = 1000 + 15 * np.sin(t.dayofyear / 365 * 2 * np.pi) + np.random.normal(0, 8)
            flowrate = 50 + 5 * np.sin(t.hour / 24 * 2 * np.pi) + np.random.normal(0, 4)
            temperature = 80 + np.random.normal(0, 2)
            vibration = 0.5 + 0.02 * np.sin(t.hour / 24 * 4 * np.pi) + np.random.normal(0, 0.04)
            rows.append([well, t, pressure, flowrate, temperature, vibration])
    df_dummy = pd.DataFrame(
        rows, columns=["well_id", "timestamp", "pressure", "flowrate", "temperature", "vibration"]
    )
    df_dummy.to_csv(cfg.input_path, index=False)
    print(f"Data sintetis dibuat: {cfg.input_path} ({len(df_dummy):,} baris)")
else:
    print(f"File sudah ada, pakai data asli: {cfg.input_path}")

In [ ]:
df_raw = load_data(cfg)
print(f"Shape: {df_raw.shape}")
df_raw.head()

In [ ]:
df_raw.info()

# Eksplorasi Awal

In [ ]:
df_raw[cfg.target_cols].describe()

In [ ]:
missing_pct = df_raw[cfg.target_cols].isna().mean().sort_values(ascending=False) * 100
missing_pct.plot(kind="bar", title="Persentase Missing Value per Parameter (sebelum cleaning)")
plt.ylabel("% missing")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
df_raw[cfg.datetime_col] = pd.to_datetime(df_raw[cfg.datetime_col], errors="coerce")
for key, g in df_raw.groupby(cfg.group_cols[0]):
    g = g.sort_values(cfg.datetime_col)
    ax.plot(g[cfg.datetime_col], g["pressure"], label=str(key), alpha=0.7, linewidth=0.8)
ax.set_title("Pressure mentah per grup (sebelum cleaning)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
corr = df_raw[cfg.target_cols].corr()
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(cfg.target_cols)))
ax.set_yticks(range(len(cfg.target_cols)))
ax.set_xticklabels(cfg.target_cols, rotation=45, ha="right")
ax.set_yticklabels(cfg.target_cols)
for i in range(len(cfg.target_cols)):
    for j in range(len(cfg.target_cols)):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=9)
plt.colorbar(im)
ax.set_title("Korelasi antar parameter (data mentah)")
plt.tight_layout()
plt.show()

corr

# Cleaning Data

In [ ]:
df = parse_and_sort(df_raw, cfg)
df = drop_duplicates(df, cfg)
print(f"Shape setelah parse & drop duplicate: {df.shape}")

In [ ]:
df = resample_per_group(df, cfg)
print(f"Shape setelah resample ke frekuensi '{cfg.freq}': {df.shape}")
print(f"Jumlah grup: {df['_group_key'].nunique()}")

In [ ]:
df = filter_sparse_groups(df, cfg)
print(f"Shape setelah filter grup terlalu sparse: {df.shape}")

In [ ]:
fig, axes = plt.subplots(1, len(cfg.target_cols), figsize=(14, 4))
for ax, col in zip(axes, cfg.target_cols):
    ax.boxplot(df[col].dropna())
    ax.set_title(col)
plt.suptitle("Distribusi tiap parameter (sebelum outlier handling)")
plt.tight_layout()
plt.show()

In [ ]:
df = handle_outliers(df, cfg)

# Cek ulang boxplot setelah outlier di-clip
fig, axes = plt.subplots(1, len(cfg.target_cols), figsize=(14, 4))
for ax, col in zip(axes, cfg.target_cols):
    ax.boxplot(df[col].dropna())
    ax.set_title(col)
plt.suptitle("Distribusi tiap parameter (setelah outlier di-clip)")
plt.tight_layout()
plt.show()

In [ ]:
df = impute_missing(df, cfg)
print(f"Total missing setelah imputasi: {df[cfg.target_cols].isna().sum().sum()}")
df.head()

# Feature Engineering

In [ ]:
df = add_time_idx(df, cfg)
df = add_calendar_features(df, cfg)
print("Kolom setelah feature engineering:", df.columns.tolist())
df[["_group_key", cfg.datetime_col, "time_idx"] + KNOWN_CALENDAR_FEATURES].head(10)

In [ ]:
check = df.groupby("_group_key")["time_idx"].agg(["min", "max", "count"])
check["expected_count"] = check["max"] - check["min"] + 1
check["is_continuous"] = check["count"] == check["expected_count"]
check

In [ ]:
fig, axes = plt.subplots(len(cfg.target_cols), 1, figsize=(14, 3 * len(cfg.target_cols)), sharex=True)
for ax, col in zip(axes, cfg.target_cols):
    for key, g in df.groupby("_group_key"):
        g = g.sort_values(cfg.datetime_col)
        ax.plot(g[cfg.datetime_col], g[col], label=str(key), alpha=0.7, linewidth=0.8)
    ax.set_ylabel(col)
    ax.legend(fontsize=8)
axes[0].set_title("Data setelah cleaning (siap untuk TFT)")
plt.tight_layout()
plt.show()

# Split (Cutoff time_idx untuk TFT)

In [ ]:
cutoffs = compute_split_cutoffs(df, cfg)
cutoffs

In [ ]:
train, val, test = make_reference_split(df, cutoffs, cfg)
print(f"Train (referensi EDA) : {train.shape}")
print(f"Val   (referensi EDA) : {val.shape}")
print(f"Test  (referensi EDA) : {test.shape}")

In [ ]:
sample_key = df["_group_key"].iloc[0]
g = df[df["_group_key"] == sample_key].sort_values("time_idx")
c = cutoffs[cutoffs["_group_key"] == sample_key].iloc[0]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(g["time_idx"], g["pressure"], color="black", linewidth=0.8)
ax.axvspan(0, c["train_cutoff_time_idx"], color="tab:blue", alpha=0.15, label="train")
ax.axvspan(c["train_cutoff_time_idx"], c["val_cutoff_time_idx"], color="tab:orange", alpha=0.15, label="val")
ax.axvspan(c["val_cutoff_time_idx"], g["time_idx"].max(), color="tab:green", alpha=0.15, label="test")
ax.set_title(f"Pembagian train/val/test - grup: {sample_key} (pressure)")
ax.set_xlabel("time_idx")
ax.legend()
plt.tight_layout()
plt.show()

# Save Output

In [ ]:
result = save_outputs(df, train, val, test, cutoffs, cfg)

with open(f"{cfg.output_dir}/metadata.json") as f:
    metadata = json.load(f)

print(json.dumps(metadata, indent=2))